## Shepherd Model Applied to Complex Pack Systems


This Code is designed to streamline the setup of complex battery geometries and show a simplified example of how they behave.
For a thorough explination of how the model calculates the state of the battery refer to the "How it works" Section at the bottom!

#### To Run This Code
First, click the $\rhd\rhd$ button on the banner at the top of the screen. This will reload the notebook and run all of the cells. This might take a minute, as this program imports a number of packages. This will load boxes under the "Inputs", "Run Iteratation", and "Graphs" sections below. No other cells will needed to be run to use this program. \
Instructions for how to use each section are down below!\
\
All of the code blocks should be compressed. If they aren't you can click on the side of them to close them!\
\
Once all of the cells have been run, go down to the Inputs section.

#### Pack Dimensions

The way the Cells are layed out is as followed. Batteries Pack are composed of strings, where each string is said to have <b>m</b> Cells:

$$
\begin{bmatrix}
C_{1,1} & C_{1,2} & ... & C_{1,m}
\end{bmatrix}
$$

Each pack will have a <b>n</b> of these strings in parrallel: <br>


$$
\begin{bmatrix}
C_{1,1} & C_{1,2} & ... & C_{1,m}\\
C_{2,1} & C_{2,2}  & & \vdots\\
\vdots& &  \ddots& \vdots\\
C_{n,1} & C_{n,2} & ... & C_{n,m}\\
\end{bmatrix}
$$

So each Battery pack is said to have n strings in parallel, each string containing m cells in series.<br>
This is depicted as such: 
$$[mS-nP] $$
This means that a pack will have $n \cdot m$ cells, each with a position labeled as $C_{i,j}$ where $i$ is the string they are in and $j$ is its position in that string.
<br>

An example:
$$
\begin{bmatrix}
C_{1,1} & \mathbf{C_{1,2}} & C_{1,3}\\
C_{2,1} & C_{2,2} & C_{2,3} \\
\end{bmatrix}
$$

The above pack has 3 cells per string and 2 strings in parallel. This is described in the following format: <br>
$$ [3S-2P] $$
While the bolded Cell $(C_{1,2})$ would be cell 1,2 where it is in the first string and is the second cell in that string. 

Each of these Cells have their own properties which each impact the Voltage!

### Code

In [1]:
# import packages
import ipywidgets as widgets
from ipywidgets import Layout
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as optimize

#Shepard Equation
def Shepherd_PS(V0, R, I, K, dx, Isum, Q, A, B):
    V = V0 - R*I - K/(1-(dx*Isum)/Q) * ((dx*Isum) + I) + A * np.exp(-B* (dx*Isum)/(Q))
    return V

In [2]:
#Itteration and other functions
#Last edited: (2/03/2026) -> Added function calling multiple variables
import numpy as np
import scipy.optimize as optimize

def Qmin(Q): #Lowest value of Q for each string
    Q_min = np.zeros(N.value)
    for i in range(0, N.value):
        Q_min[i] = min(Q[i,:])
    return Q_min

def Qb(Q): #Charge Capacity of pack/battery
    Q_min = Qmin(Q)
    Q_b = np.sum(Q_min)              
    return Q_b



def Iteration(x, V0, R, K, Q, C, x_off = np.array([False])):
    
    Q_b = Qb(Q)
    
    dx = (x[1]-x[0])/C
    Conv = np.zeros((len(x), N.value))     #Conversion Factors for every string at every SOD
    fitted_Is = np.zeros((len(x), N.value))
    Isum = np.zeros((len(x), N.value))
    I_b_test = Q_b * C               #Testing each SOD at a C-rate of 1
    V_c = np.zeros(M.value)                #Cell current in string
    V_t = np.zeros(N.value)                #Testing voltage
    
    #First determine conversion factors for each string
    #DETERMINE INITIAL GUESS -> helps speed up iteration
    
    I_fact = np.zeros(N.value)             #An array determine initial guesses for current distribution
    
    for i in range(0,len(I_fact)):
        I_fact[i] = np.sum(R[i,:])/np.sum(R)*np.min(Q[i,:])/Q_b
    Norm = np.sum(I_fact) #Normalization factor
    
    for i in range(0, len(I_fact)):
        I_fact[i] = I_fact[i]/Norm
    I_g = I_fact*I_b_test     #Inital "guessing values"
    
    #CONSTRAINTS -> ensures that the sum of all current adds to the total current
    def constraint1(I_g):
        con = I_b_test
        for i in range(0, len(I_g)):
            con = con - I_g[i]
        return con
    con1 = {'type': 'eq', 'fun': constraint1}
    
    #Bounds -> ensures guesses are between 0 and test battery current. This ensures there isn't negative current to satisfy the constraints
    b = [(0,I_b_test)]
    for i in range (1,len(I_g)):
        b.append((0,I_b_test))
    b = tuple(b)

    #iterate over every SOD
    for i in range(0, len(x)):
        for r in range(0,N.value):
            Isum[i,r] = np.sum(fitted_Is[0:i,r])
        if i == 0:
            for p in range(0,N.value):
                if x_off.all() > 0:
                    Isum[0,p] = x_off[p]*Q[0,p % Q.shape[1]]/dx
    #Optimizing sum of mean squares
        def SSE(I):                  #input of string current I at C rate i (from 0->len(C))
            for l in range(0,N.value):
                for j in range(0,M.value):
                    V_c[j] = Shepherd_PS(V0[l,j], R[l,j], I[l], K[l,j], dx, Isum[i,l], Q[l,j], A, B)   #Individual Cells in string (only 1 in this case)
                V_t[l] = np.sum(V_c)                                                                 #Note, I as an imput has to be an array
            SSE = np.sum((V_t[:]-np.average(V_t))**2)
            return SSE
    
        result = optimize.minimize(SSE, I_g, bounds = b, constraints = con1)
        if result.success:
            fitted_Is[i,:] = result.x
        
            for j in range(0, N.value):
                Conv[i, j] = fitted_Is[i,j]/I_b_test

        else:
            print("error on i {}".format(i))
            print(Isum)
            raise ValueError(result)
        
        if i == 0:
            for p in range(0,N.value):
                if x_off.all() > 0:
                    fitted_Is[0,p] = x_off[p]*Q[0,p % Q.shape[1]]/dx

    return Conv, fitted_Is, Isum

In [3]:
#Libraries
#Need to pick battery type
#need to pick a preset
#Need to add Weak Cell to preset
#Updated on 4.19.2026

Stepdict = {
    "V0": 0.1,
    "R" : 0.01,
    "Q" : 0.1,
    "K" : 0.01,
    },

#Place Holder Values !!!!
Batterydict = {
    "Test Values" : {
    "A" : 0.1,
    "B" : 3,
    "V0": 3.6,
    "R" : 0.03,
    "Q" : 2.25,
    "K" : 0.01,
    },
    
    "LFP": {
    "A" : 0.35,
    "B" : 1000,
    "V0": 3.3,      #V_cutoff 2.5 V
    "R" : 0.013,
    "Q" : 11,
    "K" : 0.0015,
    },
    
    "NMH": {
    "A" : 0.15,
    "B" : 17,
    "V0": 1.28,
    "R" : 0.035,
    "Q" : 2,
    "K" : 0.0035,
    },

    #"LiFeS2 [WIP]": {
    #"A" : 3,
    #"B" : 3,
    #"V0": 3,
    #"R" : 3,
    #"Q" : 3,
    #"K" : 3,
    #},
}

#Preset Dictionary for Preset Widget
Presetdict = {
  "1S-1P" : {
    "N" : 1,
    "M" : 1,
    "Pack Number" : 1,
    "#C": 1,
    "Weak": 0,  #1 = Yes, 0 = No     
  },
    
  "1S-1P : C-rate analysis" : {
    "N" : 1,
    "M" : 1,
    "Pack Number" : 1,
    "#C": 4,
    "Weak": 0,
      
  },

  "1S-2P - Weak R" : {
    "N" : 2,
    "M" : 1,
    "Pack Number" : 1,
    "#C": 1,
    "Weak": 1,  #1 = Yes, 0 = No
    "R" : 2, #is multiplied by battery chem value so 1.1 is 110% of the normal value
    "Q" : None,      
  },
    
  "1S-2P - Weak Q" : {
    "N" : 2,
    "M" : 1,
    "Pack Number" : 1,
    "#C": 1,
    "Weak": 1,  #1 = Yes, 0 = No
    "R" : None,
    "Q" : 0.5,      
  },

  "2S-2P - Weak R" : {
    "N" : 2,
    "M" : 2,
    "Pack Number" : 1,
    "#C": 1,
    "Weak": 1,  #1 = Yes, 0 = No
    "R" : 2,
    "Q" : None      
  },

  "2S-2P - Weak Q" : {
    "N" : 2,
    "M" : 2,
    "Pack Number" : 1,
    "#C": 1,
    "Weak": 1,  #1 = Yes, 0 = No
    "R" : None,
    "Q" : 0.5,      
  },
}


Param_labels = (["V0 - Nominal Voltage [V]", 
           "R - Resitance [Ohms]", 
           "Q - Charge Capacity [Ahr]", 
           "K - Polarization Constant [Ohms or Ohms/hr]"])

Autopop_Param_Map = {
    0: 'V0',  # replace with your actual param names
    1: 'R',
    2: 'Q',
    3: 'K',

}

type_keys = []
for key in Batterydict:
    type_keys.append(key)

Preset_keys = []
for key in Presetdict:
    Preset_keys.append(key)


In [4]:
#Functions for building Widgets

_rebuilding = False

def autofill(change):
    for i in range(0, len(autopop)):
        for ii in range(0, len(Widgdict['V0'])):
            Widgdict[Param_Keys[i]][ii].value = autopop[i].value

# Container that will hold the grid
grid_output = widgets.Output()

def build_grid(*args):      #Function to automatically build the widgets

    if _rebuilding:
        return  # Skip if a batch update is in progress

    with grid_output:
        grid_output.clear_output()
        items = []
        gridDis = []
        labels = []
        Size = []

        for k in Widgdict:             #Making a widget for each cell
            Widgdict[k] = [
                widgets.FloatText(
                    description=f'{k} for Cell {i+1},{j+1}',
                    layout=Layout(width='150px'),
                    style = {'description_width': 'initial'},
                    value = autopop[len(Size)].value,
                    step = Stepdict[0][k],
                )
                for i in range(N.value)
                for j in range(M.value)
            ]
            if Presetdict[Preset_keys[Preset.value]]['Weak'] == 1:
                if Presetdict[Preset_keys[Preset.value]]['R']:
                    Widgdict["R"][0].value = Presetdict[Preset_keys[Preset.value]]['R']*Batterydict[type_keys[Type.value]]['R']

            if Presetdict[Preset_keys[Preset.value]]['Weak'] == 1:    
                if Presetdict[Preset_keys[Preset.value]]['Q']:
                    Widgdict["Q"][0].value = Presetdict[Preset_keys[Preset.value]]['Q']*Batterydict[type_keys[Type.value]]['Q']
            #print(Widgdict)
            for w in Widgdict[k]:       # Add reset trigger
                w.observe(reset_results, names='value')
            items.append(Widgdict[k])   #each set of widegts gets added into a list: one set per variable
            Size.append(k)              #A list to select autopopulate value

        for i in range(0,len(Param_Keys)):    #Creating a grid for each Variable Widget
            grid = widgets.GridBox(
                items[i],
                layout = widgets.Layout(
                    grid_template_columns=f"repeat({M.value}, 150px)",
                    grid_gap="10px"
                )
            )
            gridDis.append(grid)

        for i in range(0, len(Param_Keys)):
            labeltxt = widgets.HTML(
                value= f"  <b>{Param_labels[i]}</b>",
                #placeholder='Some HTML',
                description='Variable:',
            )
            labels.append(labeltxt)
        
        for i in range(0, len(Param_Keys)):
            display(labels[i], autopop[i], gridDis[i])

def C_bounds(change):      
    with C_bns:
        C_bns.clear_output()
        if C_num.value > 1:
            display(C_lower_bound, C_upper_bound)
            C_rate.value = 3.6
        if C_num.value == 1:
            display(C_rate)
            C_lower_bound.value = 1
            C_upper_bound.value = 3

def C_bnd_adj(Change):
    C_lower_bound.max = C_upper_bound.value - 0.2

def C_rate_build(*args):
    with C_bns:
        C_bns.clear_output()
        display(C_rate)

#Change Pack Configuration, C-rate, number of Packs
def Preset_change(change):
    PackNum.value = Presetdict[Preset_keys[Preset.value]]["Pack Number"]
    N.value = Presetdict[Preset_keys[Preset.value]]["N"]
    M.value = Presetdict[Preset_keys[Preset.value]]["M"]
    C_num.value = Presetdict[Preset_keys[Preset.value]]["#C"]
    if Presetdict[Preset_keys[Preset.value]]['Weak'] == 1:
        if Presetdict[Preset_keys[Preset.value]]['R']:
            Widgdict["R"][0].value = Presetdict[Preset_keys[Preset.value]]['R']*Batterydict[type_keys[Type.value]]['R']
        if Presetdict[Preset_keys[Preset.value]]['Q']:
            Widgdict["Q"][0].value = Presetdict[Preset_keys[Preset.value]]['Q']*Batterydict[type_keys[Type.value]]['Q']
    else:
        Widgdict["R"][0].value = Batterydict[type_keys[Type.value]]['R']
        Widgdict["Q"][0].value = Batterydict[type_keys[Type.value]]['Q']
    build_grid()

#Reset the iteration if the widgets are changed
def reset_results(change):
    global iteration_result
    iteration_result = None

def Type_change(change):
    global _rebuilding
    _rebuilding = True  # Block cascading build_grid calls
    try:
        # Update autopop values (these will fire observe, but build_grid will skip)
        for i in range(0, len(Param_Keys)):
            autopop[i].value = Batterydict[type_keys[Type.value]][Param_Keys[i]]
        a.value = Batterydict[type_keys[Type.value]]['A']
        b.value = Batterydict[type_keys[Type.value]]['B']
    finally:
        _rebuilding = False  # Always release the flag, even if an error occurs
    build_grid()  # Single rebuild at the end


def V_cutoff_change(change):
    V_cut_off_Widget.value = autopop[0].value*0.5




### Inputs

Here you will pick the desired Chemistry and Preset. Each of these are optional but can streamline the setup process. These determine the Cell Properties (A, B, V0, R, Q, and K) and Pack Geometry (N, M, I), as well as weaknesses, such as low Charge Capacities or high resitances. \
Once you've picked from these options you can plug in your own values into every Field: \
\
$\hspace{10pt}$ $\textbf{N}$ - number of parallel strings in the battery pack. 1 represents a single string. \
$\hspace{10pt}$ $\textbf{M}$ - number of cells per strings in the battery pack. \
$\hspace{10pt}$ $\textbf{Number of Currents}$ - number of Currents the system analyzes. \
$\hspace{10pt}$ $\textbf{Min/Max Currents}$ - Range of Currents. \
$\hspace{10pt}$ $\textbf{Paramaters}$ - $A, B, V_0, R, Q,$ and $K$. These all represent different cell properties in the Shepherd Equation.

There are a number of presets designed to highlight different properties of these configurations, they should help highlight how the model works! \
Once you have set up the system move down to the "Run Iteratation" section.

<b>Note: </b> This system does not handel large differences in voltage well. In reality, cells with large enough voltage difference would end up charging each other, but that isn't reperesented in this model! You can compare what happens when you put a large voltage difference to normal behavior!

In [5]:
#Display and observe Widgets
#Widgets are listed in the order that they Appear

#
# Moddeling Conditions: Material Type, Battery Arrangement, # of Packs (Not currently used) 
Type = widgets.Dropdown(
    options = [(type_keys[i], i) for i in range(0, len(type_keys))],
    value = 0,
    layout=Layout(width='210px'),
    description = "Battery Chemistry",
    style = {'description_width': 'initial'},
)
Preset = widgets.Dropdown(
    options = [(Preset_keys[i], i) for i in range(0, len(Preset_keys))],
    value = 0,
    layout=Layout(width='210px'),
    description = "Model Preset",
    style = {'description_width': 'initial'},
)

#
#Not Yet implemented
#
PackNum = widgets.BoundedIntText(
    value=1,
    min=1,
    max=4,
    step=1,
    layout=Layout(width='225px'),
    style = {'description_width': 'initial'},
    description = 'Number of Packs [Min:1, Max 4]',
    disabled = False,
)


#Battery Pack Dimensions
N = widgets.BoundedIntText(
    min = 1,
    description= '$N$ - # of Parallel Strings',
    layout=Layout(width='210px'),
    style = {'description_width': 'initial'},
    value = Presetdict[Preset_keys[Preset.value]]["N"]
)
M = widgets.BoundedIntText(
    min = 1,
    description= '$ M $ - # of Cells per String',
    layout=Layout(width='210px'),
    style = {'description_width': 'initial'},
    value = Presetdict[Preset_keys[Preset.value]]["M"]
)  


# C-rate Number and Bounds
C_num = widgets.BoundedIntText(
    description= 'Number of Currents',
    layout=Layout(width='210px'),
    style = {'description_width': 'initial'},
    value = Presetdict[Preset_keys[Preset.value]]["#C"],
    min = 1,
    max = 5
)

C_rate = widgets.FloatText(
                    description= 'Single Current [Amps]',
                    layout=Layout(width='210px'),
                    style = {'description_width': 'initial'},
                    value = 3.6,
                    readout_format='.3f',
                )

C_upper_bound = widgets.BoundedFloatText(
                    description= 'Largest Current [Amps]',
                    layout=Layout(width='210px'),
                    style = {'description_width': 'initial'},
                    value = 3,
                    min= 3,
                    max = 50,    
                )
C_lower_bound = widgets.BoundedFloatText(
                    description= 'Smallest Current [Amps]',
                    layout=Layout(width='210px'),
                    style = {'description_width': 'initial'},
                    value = 1,
                    min=0.2,
                    max = C_upper_bound.value - 0.2,
                ) 

#Battery Params
a = widgets.FloatText(
                    description= 'A - Fitting Variable [V]',
                    layout=Layout(width='210px'),
                    style = {'description_width': 'initial'},
                    value = Batterydict[type_keys[Type.value]]['A']
                )
b = widgets.FloatText(
                    description= 'B - Fitting Variable',
                    layout=Layout(width='210px'),
                    style = {'description_width': 'initial'},
                    value = Batterydict[type_keys[Type.value]]['B']
                )

# Dictionary to store widgets
Widgdict = {
    "V0": None,
    "R": None,
    "Q": None,
    "K": None
}

Param_Keys = []      # list for storing Paramaters
for key in Widgdict:
    Param_Keys.append(key)  #adding all of the paramater names from "Widgdict"

autopop = []         # list for autopopulate widgets which adjust Paramater Values
for i in range(0, len(Param_Keys)):    
    auto = widgets.FloatText(
            description= f'Auto Fill {Param_Keys[i]}',
            layout=Layout(width='200px'),
            style = {'description_width': 'initial'},
            value = Batterydict[type_keys[Type.value]][Param_Keys[i]],
            step = Stepdict[0][Param_Keys[i]],       
    )
    autopop.append(auto)  #Generating Autopopulate widget and adding to the list


#C-rate Box edits
C_bns = widgets.Output()


# Dictionary to store Paramaters
Paramdict = {}
#Getting Parameters for iteration
def get_parameters():
    for k in Widgdict:
        Paramdict[k] = np.array([
            w.value for w in Widgdict[k]
        ]).reshape(N.value, M.value)
    return Paramdict


#Trigers when Any Widgets are changed to update grid
Type.observe(Type_change, names = 'value')
Preset.observe(Preset_change, names = 'value')
C_num.observe(C_bounds, names = 'value')
C_upper_bound.observe(C_bnd_adj, names = 'value')
N.observe(build_grid, names='value')
M.observe(build_grid, names='value')
for i in range(0, len(Param_Keys)):
    autopop[i].observe(autofill, names= "value")


# Iteration Reset
Type.observe(reset_results, names='value')
Preset.observe(reset_results, names='value')
C_num.observe(reset_results, names='value')
C_rate.observe(reset_results, names='value')
C_upper_bound.observe(reset_results, names='value')
C_lower_bound.observe(reset_results, names='value')
N.observe(reset_results, names='value')
M.observe(reset_results, names='value')
a.observe(reset_results, names='value')
b.observe(reset_results, names='value')
for i in range(0, len(Param_Keys)):
    autopop[i].observe(reset_results, names= "value")

#Adjusting V cut off
autopop[0].observe(V_cutoff_change, names = "value")
M.observe(V_cutoff_change, names = "value")

build_grid()    # Initial build

# Labels
Conditions = widgets.HTML(
    value= f"  <b>Model Conditions: Select Battery Chemistry and a Preset</b> <br> <b>Note:</b> Unique values can all be entered below, presets are meant to be a starting point!")
Dimension = widgets.HTML(
    value= f"  <b>Dimension of Battery Pack: </b> Change the number of Strings and Cells per String")
C_rate_label = widgets.HTML(
    value= f"  <b> System Currents, defined by C-rate [hr<sup>-1</sup>]: </b>If multiple C-rates, there will be a range between the min and max C-rate")
Paramaters = widgets.HTML(
    value= f"  <b>System Paramaters:</b> Properties of the system that impact behavior.")


#PackNum, Add after Preset ???

#Display Widgets and Labels
display(Conditions, Type, Preset,  Dimension, N, M, C_rate_label, C_num, C_bns)
C_rate_build()
if C_num.value > 1:
    display(C_lower_bound, C_upper_bound)
display(Paramaters, a, b, grid_output)


HTML(value='  <b>Model Conditions: Select Battery Chemistry and a Preset</b> <br> <b>Note:</b> Unique values c…

Dropdown(description='Battery Chemistry', layout=Layout(width='210px'), options=(('Test Values', 0), ('LFP', 1…

Dropdown(description='Model Preset', layout=Layout(width='210px'), options=(('1S-1P', 0), ('1S-1P : C-rate ana…

HTML(value='  <b>Dimension of Battery Pack: </b> Change the number of Strings and Cells per String')

BoundedIntText(value=1, description='$N$ - # of Parallel Strings', layout=Layout(width='210px'), min=1, style=…

...

Output()

HTML(value='  <b>System Paramaters:</b> Properties of the system that impact behavior.')

FloatText(value=0.1, description='A - Fitting Variable [V]', layout=Layout(width='210px'), style=DescriptionSt…

FloatText(value=3.0, description='B - Fitting Variable', layout=Layout(width='210px'), style=DescriptionStyle(…

Output()

## Run Iteratation
Once all of the variables are assigned, you can begin the iteration. The "Run Iteration" section is designed to let you set the Number of Points to analyze and the State of Discharge [SOD or x] starting/ending point. These determine the bounds of the calculation. \
You should never need to change the number of points, but if your computer is running slowly then you can turn it down to 100 without loosing detail. \
The point of the bounds for SOD is to analyze the impact of unbalanced cells in a pack. An example can be an analysis from 20% SOD to 80% SOD. This can be set with the values 0.2 and 0.8 respectively. Most of the time none of these boxes need to be changed! \
Once you are ready, click the Initiate Iteration button. A popup will appear once it is done complete. If you change any values in the "Input" section then you will have to rerun the iteration.

<b>Note: </b> The larger and more complex the system becomes (big N and M values, varried paramaters) the longer iteration can take. Running a [10S-10P] system might take a while!

In [6]:
# Run Iteration

x_index = widgets.IntText(
                    description= 'Number of Indexes',
                    layout=Layout(width='225px'),
                    style = {'description_width': 'initial'},
                    value = 1000,

                ) 

x_lower_bound = widgets.BoundedFloatText(
                    description= 'SOD starting point',
                    layout=Layout(width='225px'),
                    style = {'description_width': 'initial'},
                    value = 0,
                    min = 0,
                    max = 0.8,
                    step = 0.1,
                ) 

x_upper_bound = widgets.BoundedFloatText(
                    description= 'SOD ending point',
                    layout=Layout(width='225px'),
                    style = {'description_width': 'initial'},
                    value = 0.999,
                    min = 0.2,
                    max = 0.999,
                    step = 0.1,
                ) 


# button to Iterate:
iterate_button = widgets.Button(
    description='Initiate Iteration',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click me',
    icon='check' # (FontAwesome names without the `fa-` prefix)
)

In_Progress = widgets.HTML(
    value= f"  <b>Iteration In Progress...</b>")

Complete = widgets.HTML(
    value= f"  <b>Iteration Complete!</b>")

##
## Add output to function
##
Iterate_out = widgets.Output()


def Button_iterate(btn):
    global iteration_result
    global standard_result
    global A
    global B
    global t
    global t_norm

    iteration_result = []
    standard_result = []
    t = []
    t_norm = []
    

    
    params = get_parameters()
    V0 = params["V0"]
    R = params["R"]
    Q = params["Q"]
    K = params["K"]
    A = a.value
    B = b.value

    V_norm = np.ndarray((N.value,M.value))
    R_norm = np.ndarray((N.value,M.value))
    Q_norm = np.ndarray((N.value,M.value))
    K_norm = np.ndarray((N.value,M.value))
    
    V_norm[:] = autopop[0].value
    R_norm[:] = autopop[1].value
    Q_norm[:] = autopop[2].value
    K_norm[:] = autopop[3].value
  
    x = np.linspace(x_lower_bound.value, x_upper_bound.value, x_index.value)
    if C_num.value == 1:
        Currents = np.array([C_rate.value])
        
        
    if C_num.value > 1:
        Currents = np.linspace(C_lower_bound.value, C_upper_bound.value, C_num.value)

    C_rates = Currents/Qb(Q)
    C_rates_norm = Currents/Qb(Q_norm)
    
    
    for i in C_rates:
        t.append(np.linspace(0, 1/i, x_index.value))
    for i in C_rates_norm:
        t_norm.append(np.linspace(0, 1/i, x_index.value))
    
    x_off = np.zeros(N.value)
    if x_lower_bound.value > 0:
        for i in range(0, N.value):
            x_off[i] = x_lower_bound.value
    
    with Iterate_out:
        Iterate_out.clear_output()
        display(In_Progress)
        for i in range(0, len(C_rates)):
            #print(Q) #For Diagnostic purposes, commented when not testing
            iteration_result.append(Iteration(x, V0, R, K, Q, C_rates[i], x_off))
        standard_result.append(Iteration(x, V_norm, R_norm, K_norm, Q_norm, C_rates_norm[0], x_off))

        Complete = widgets.HTML(
            value= f"  <b>Iteration Complete!</b>")
        display(Complete)

x_index.observe(reset_results, names='value') # Reset iteration if variables change
x_lower_bound.observe(reset_results, names='value')
x_upper_bound.observe(reset_results, names='value')


iterate_button.on_click(Button_iterate)


WARNING = 'DO NOT make the SOD ending point 1, it will break the code! Do 0.999 instead'

display(x_index, x_lower_bound, x_upper_bound, WARNING, iterate_button, Iterate_out)

IntText(value=1000, description='Number of Indexes', layout=Layout(width='225px'), style=DescriptionStyle(desc…

BoundedFloatText(value=0.0, description='SOD starting point', layout=Layout(width='225px'), max=0.8, step=0.1,…

BoundedFloatText(value=0.999, description='SOD ending point', layout=Layout(width='225px'), max=0.999, min=0.2…

'DO NOT make the SOD ending point 1, it will break the code! Do 0.999 instead'

Button(description='Initiate Iteration', icon='check', style=ButtonStyle(), tooltip='Click me')

Output()

### Graphs
The last section lets you pick any number of graphical outputs from the selection. Once you have selected them, click "Plot the Selection" and all of the graphs will be shown. Reclicking the button with update the graphs. The names are hopefully self-explanitory, but a short description of each is given below: \
\
<b>Voltage Cutoff: </b> Displays a line on all voltage graphs. This value is based on the "Battery Chemistry" values, so it might need manual adjustment. \
<b>SOD or Time: </b> This option Changes the x-axis of the graphs between SOD and Time. Each are useful to compare different aspects of the system! 


<b>Current Distribution: </b> This option will graph all strings and show how much of the battery current they are taking on. \
<b>String Charge vs Pack SOD: </b> This shows how much charge each string has against the pack SOD. \
<b>Individual Cell Voltages:</b> This displays individual cell voltages at the lowest current. if M = 1 then only 1 line should be visable (unless the paramaters break the interation!) \
<b>Impact of Current on Pack Voltage:</b> This shows 1 line per Current. Each line repersents the system under that current.
<b>Pack vs Standard Pack:</b> This takes the autofill values to create an 'Ideal' pack. Note: Best viewed with Time as the x-axis to properly see differences in Pack behavior.

In [7]:
#Ploting Results
#Has the functions for each button. These should probably be compressed into a single "Graph Function" that takes different inputs, 
#but as of the moment each result is independantly created.

# Graphing paramaters
titlesize = 16
axissize = 14

markers = np.array([None, "o", "s", "^", "x", "1", "p", "+"]) # Marker per C-rate
mark_every = x_index.value/10
marker_size = 5

color_palate_1 = np.array(["#FFB000", "#648FFF", "#FE6100", "#785EF0", "#DC267F"]) # 1 color per string
linestyles = np.array(["solid", "dashed", "dashdot", "dotted"]) # Linestyle per Cell in string

line_width = 1.5

V_cut_off_Widget = widgets.BoundedFloatText(
                    description= 'Voltage Cutoff',
                    layout=Layout(width='225px'),
                    style = {'description_width': 'initial'},
                    value = Batterydict[type_keys[Type.value]]["V0"]*0.5,
                    max = Batterydict[type_keys[Type.value]]["V0"],
                ) 

T_SOD_Switch = widgets.Dropdown(
    options = [('State of Discharge',0),('Time',1)],
    value = 0,
    layout=Layout(width='225px'),
    description = "SOD or Time",
    style = {'description_width': 'initial'},
)

#Checklist of options:
Current_Dist = widgets.Checkbox(
    value=False,
    description='Current Distribution in each String [All Currents]',
    disabled=False,
    indent=False
)

Charge_SOD = widgets.Checkbox(
    value=False,
    description='String Charge vs Pack SOD for each String [All Currents]',
    disabled=False,
    indent=False
)



Discharge_cells = widgets.Checkbox(
    value=False,
    description='Individual Cell Voltages [Only Shows lowest Current]',
    disabled=False,
    indent=False
)

Discharge_C_rate = widgets.Checkbox(
    value=False,
    description='Impact of Current on Pack Voltage',
    disabled=False,
    indent=False
)

Discharge_Pack = widgets.Checkbox(
    value=False,
    description='Pack vs Standard Pack',
    disabled=False,
    indent=False
)

# button to run:
plot_button = widgets.Button(
    description='Plot the Selection',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click me',
    icon='check' # (FontAwesome names without the `fa-` prefix)
)


output = widgets.Output()

def plot_fun(btn):
    #Need to check which options are clicked
    with output:
        output.clear_output()

        if 'iteration_result' not in globals():
            print("Iteration has not been run, go up to the Iteration Trigger section!")
            return
        
        if iteration_result is None:
            print("Variables have been changed, rerun the Iteration above!")
            return
        
        # Recreate x safely
        x = np.linspace(x_lower_bound.value,
                        x_upper_bound.value,
                        x_index.value)

        params = get_parameters()
        V0 = params["V0"]
        R = params["R"]
        Q = params["Q"]
        K = params["K"]
        A = a.value
        B = b.value

        #Maybe Functionalize this!
        V_norm = np.ndarray((N.value,M.value))
        R_norm = np.ndarray((N.value,M.value))
        Q_norm = np.ndarray((N.value,M.value))
        K_norm = np.ndarray((N.value,M.value))
    
        V_norm[:] = autopop[0].value
        R_norm[:] = autopop[1].value
        Q_norm[:] = autopop[2].value
        K_norm[:] = autopop[3].value

        if C_num.value == 1:
            Currents = np.array([C_rate.value])
        
        
        if C_num.value > 1:
            Currents = np.linspace(C_lower_bound.value, C_upper_bound.value, C_num.value)

        C_rates = Currents/Qb(Q)
        C_rates_norm = Currents/Qb(Q_norm)

        #
        #Editing this rn
        #

        ### 
        ###
        ### Current Distribution Graph
        if Current_Dist.value:
            for c in range(0, len(C_rates)): #
                for i in range(0, N.value):
                    if T_SOD_Switch.value == 1:
                        plt.plot(t[c], iteration_result[c][0][:,i],  marker = markers[0], markevery = int(mark_every),
                                 markersize = marker_size, linestyle = linestyles[c], color = color_palate_1[i % len(color_palate_1)], linewidth = line_width, 
                                 label = "C-rate {:.1f}, String {:.0f}".format(C_rates[c], i+1)) #
                        #markers color_palate_1 linestyles
                    else:
                        plt.plot(x, iteration_result[c][0][:,i],  marker = markers[0], markevery = int(mark_every), 
                                 markersize = marker_size, linestyle = linestyles[c], color = color_palate_1[i % len(color_palate_1)], linewidth = line_width,
                                 label = "C-rate {:.1f}, String {:.0f}".format(C_rates[c], i+1))
            
            plt.legend(loc='upper left', bbox_to_anchor=(1, 1)), 
            plt.title("Distribution of Currents in Each String", fontsize = titlesize)

            plt.ylabel("Fraction of Pack Current in each string", fontsize = axissize)
            if T_SOD_Switch.value == 1:
                plt.xlim(0, t[0][-1])
                plt.xlabel("Time [Hours]", fontsize = axissize)
            else:
                plt.xlim(0,1.01)
                plt.xlabel("Pack State of Discharge [x]", fontsize = axissize)
            plt.ylim(0,1.01)
            plt.grid(linewidth = '0.2')
            plt.show()


        ###
        ### 
        ###
        if Charge_SOD.value:

            params = get_parameters()
            Q = params["Q"]
            Q_mod = Qmin(Q)
            
            Q_plot = np.zeros((C_num.value, N.value,len(x)))
            I_tot = np.zeros((C_num.value, N.value,len(x)))
            QDiff = np.zeros((C_num.value, N.value,len(x)))

            #Create Data
            for c in range(0, len(C_rates)):
                for i in range(0,N.value):
                    for j in range(0,len(x)):
                        I_tot[c,i,j] = iteration_result[c][0][j,i]*(x[1]-x[0])*Qb(Q)
                        QDiff[c,i,j] = (Q_mod[i] - np.sum(I_tot[c,i,0:j]))/Q_mod[i]

            #Plot Data
                    if T_SOD_Switch.value == 1:
                        plt.plot(t[c], QDiff[c, i,:],  marker = markers[0], markevery = int(mark_every), 
                                 markersize = marker_size, linestyle = linestyles[c], linewidth = line_width, 
                                 color = color_palate_1[i % len(color_palate_1)], label = "C-rate {:.1f}, String {:.0f}".format(C_rates[c], i+1))
                    else:
                        plt.plot(x,QDiff[c, i,:],  marker = markers[0], markevery = int(mark_every), 
                                 markersize = marker_size, linestyle = linestyles[c], linewidth = line_width, 
                                 color = color_palate_1[i % len(color_palate_1)], label = "C-rate {:.1f}, String {:.0f}".format(C_rates[c], i+1))
                        
            plt.title("Charge of each string over state of discharge", fontsize = titlesize)
            plt.ylabel("String State of Charge [1-x]", fontsize = axissize)
            plt.grid(linewidth = '0.2')
            if T_SOD_Switch.value == 1:
                plt.xlim(0, t[0][-1])
                plt.xlabel("Time [Hours]", fontsize = axissize)
            else:
                plt.xlim(0,1.01)
                plt.xlabel("Pack State of Discharge [x]", fontsize = axissize)
            plt.ylim(0,1.01)
            plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
            plt.show()
            
        ###
        ### 
        ###
        if Discharge_cells.value:

            V_cells_i = np.zeros((len(x), N.value, M.value))
            dx = x[1] - x[0]
            
            #Creating an array of voltages
            for k in range(0,len(x)):   #@ every state of charge (based on weakest cell)
                for i in range(0,N.value):
                    for j in range(0,M.value):    #@ every cell, then summed
                        #V_c[j] = Shepherd_PS(V0[l,j], R[l,j], iteration_result[1][k, l], K[l,j], dx, iteration_result[2][k,l], Q[l,j], A, B)
                        V_cells_i[k, i, j] = Shepherd_PS(V0[i,j], R[i,j], iteration_result[0][1][k, i], K[i,j], dx/C_rates[0], iteration_result[0][2][k,i], Q[i,j], A, B)
            
            #Plotting Voltages for all C values
            for i in range(0, N.value):
                for j in range(0, M.value):
                    if T_SOD_Switch.value == 1:
                        plt.plot(t[0][1:], V_cells_i[1:,i, j], marker = markers[j % len(markers)], markevery = int(mark_every), 
                                 markersize = marker_size, linestyle = linestyles[0], linewidth = line_width, 
                                 color = color_palate_1[i % len(color_palate_1)], label = "String {:.0f}, Cell {:.0f}".format(i+1, j+1))
                    else:
                        plt.plot(x[1:], V_cells_i[1:,i, j], marker = markers[j % len(markers)], markevery = int(mark_every), 
                                 markersize = marker_size, linestyle = linestyles[0], linewidth = line_width, 
                                 color = color_palate_1[i % len(color_palate_1)], label = "String {:.0f}, Cell {:.0f}".format(i+1, j+1))
            
            
            
            #Plot Voltage Cutoff
            if T_SOD_Switch.value == 1:
                plt.hlines(V_cut_off_Widget.value, 0, t[0][-1], color = "r", linestyle = "--", label = "Voltage Cutoff")
            else:
                plt.hlines(V_cut_off_Widget.value, 0, 1.01,  color = "r", linestyle = "--", label = "Voltage Cutoff")


            # x axis title and 
            if T_SOD_Switch.value == 1:
                plt.xlim(0,t[0][-1])
                plt.xlabel("Time [Hours]", fontsize = axissize)
            else:
                plt.xlim(0,1.01)
                plt.xlabel("State of Discharge [x]", fontsize = axissize)
            plt.ylabel("Voltage [V]", fontsize = axissize)
            plt.grid(linewidth = 0.2)
            plt.title("Individual Cells at {:.1f} C-rate".format(C_rates[0]), fontsize = titlesize)
            plt.ylim(0,V_cells_i[1,0,0]+1)
            plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
            plt.show()

        ###
        ###
        ###
        if Discharge_C_rate.value:

            #V_c = np.zeros(M.value)
            V_cells = np.zeros((C_num.value, len(x), N.value, M.value))
            V_s = np.zeros((C_num.value, len(x),N.value))
            dx = x[1] - x[0]
            
            #Creating an array of voltages
            for c in range(0,len(C_rates)):
                for k in range(0,len(x)):   #@ every state of charge (based on weakest cell)
                    for i in range(0, N.value):
                        for j in range(0, M.value):    #@ every cell, then summed
                            V_cells[c, k, i, j] = Shepherd_PS(V0[i,j], R[i,j], iteration_result[c][1][k, i], K[i,j], dx/C_rates[c], iteration_result[c][2][k,i], Q[i,j], A, B)
                        V_s[c, k,i] = np.sum(V_cells[c, k,i,:])

            #Plotting Voltages for all C values
            for c in range(0, len(C_rates)):
                if T_SOD_Switch.value == 1:
                    plt.plot(t[c][1:], V_s[c,1:,0], marker = markers[0], markevery = int(mark_every), 
                             markersize = marker_size, linestyle = linestyles[c], linewidth = line_width, 
                             color = color_palate_1[0], label = "Pack voltage, {:.1f}C".format(C_rates[c]))
                else:
                    plt.plot(x[1:], V_s[c,1:,0], marker = markers[0], markevery = int(mark_every), 
                             markersize = marker_size, linestyle = linestyles[c], linewidth = line_width, 
                             color = color_palate_1[0], label = "Pack voltage, {:.1f}C".format(C_rates[c]))

            #Plot Voltage Cutoff
            if T_SOD_Switch.value == 1:
                plt.hlines(V_cut_off_Widget.value*M.value, 0, t[0][-1], color = "r", linestyle = "--", label = "Voltage Cutoff")
            else:
                plt.hlines(V_cut_off_Widget.value*M.value, 0, 1.01,  color = "r", linestyle = "--", label = "Voltage Cutoff")

            #Change x axis and label
            if T_SOD_Switch.value == 1:
                plt.xlim(0, t[0][-1])
                plt.xlabel("Time [Hours]", fontsize = axissize)
            else:
                plt.xlim(0,1.01)
                plt.xlabel("State of Discharge [x]", fontsize = axissize)
                
            plt.ylabel("Voltage [V]", fontsize = axissize)
            plt.grid(linewidth = 0.2)
            plt.title("Packs under Different C-rates".format(M.value,N.value), fontsize = titlesize)
            plt.ylim(0,V_s[0,1,0]+1)
            plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
            plt.show()

        ###
        ### Working
        ###
        if Discharge_Pack.value:

            V_cells_norm = np.zeros((len(x), N.value, M.value))
            V_cells = np.zeros((len(x), N.value, M.value))
            V_s = np.zeros((len(x),N.value))
            V_s_norm = np.zeros((len(x),N.value))
            dx = x[1] - x[0]
            
            #Creating an array of voltages
            for k in range(0,len(x)):   #@ every state of charge (based on weakest cell)
                for l in range(0,N.value):
                    for j in range(0,M.value):    #@ every cell, then summed
                        V_cells[k, l, j] = Shepherd_PS(V0[l,j], R[l,j], iteration_result[0][1][k, l], K[l,j], dx/C_rates[0], iteration_result[0][2][k,l], Q[l,j], A, B)
                        V_cells_norm[k, l, j] = Shepherd_PS(V_norm[l,j], R_norm[l,j], standard_result[0][1][k, l], K_norm[l,j], dx/C_rates_norm[0], standard_result[0][2][k,l], Q_norm[l,j], A, B)
                    V_s[k,l] = np.sum(V_cells[k,l,:])
                    V_s_norm[k,l] = np.sum(V_cells_norm[k,l,:])

            
            #Plotting Voltages for both packs
            if T_SOD_Switch.value == 1:
                for i in range(0, N.value):
                    plt.plot(t[0][1:], V_s[1:,i], marker = markers[0], markevery = int(mark_every), 
                             markersize = marker_size, linestyle = linestyles[0], linewidth = line_width, color = color_palate_1[i], 
                             label = "Pack Volatge at C-rate {:.1f}".format(C_rates[0]))
                plt.plot(t_norm[0][1:], V_s_norm[1:,i],color = "black",label = "Standard Pack Volatge at C-rate {:.1f}".format(C_rates_norm[0]))
            else:
                for i in range(0, N.value):
                    plt.plot(x[1:], V_s[1:,i], marker = markers[0], markevery = int(mark_every), 
                             markersize = marker_size, linestyle = linestyles[0], linewidth = line_width, color = color_palate_1[i], label = "String {:.0f} Volatge at C-rate {:.1f}".format(i+1,C_rates[0]))
                plt.plot(x[1:], V_s_norm[1:,i], color = "black", label = "Standard Pack Volatge at C-rate {:.1f}".format(C_rates_norm[0]))

            if T_SOD_Switch.value == 1:
                plt.hlines(V_cut_off_Widget.value*M.value, 0, max(t[0][-1], t_norm[0][-1]), color = "r", linestyle = "--", label = "Voltage Cutoff")
            else:
                plt.hlines(V_cut_off_Widget.value*M.value, 0, 1.01,  color = "r", linestyle = "--", label = "Voltage Cutoff")
            
            if T_SOD_Switch.value == 1:
                plt.xlim(0, max(t[0][-1],t_norm[0][-1]))
                plt.xlabel("Time [Hours]", fontsize = axissize)
            else:
                plt.xlim(0, 1.01)
                plt.xlabel("State of Discharge [x]", fontsize = axissize)
            plt.ylabel("Voltage [V]", fontsize = axissize)
            plt.grid(linewidth = 0.2)
            plt.title("System vs Ideal Pack", fontsize = titlesize )
            plt.ylim(0,V_s[1,0]+1)
            plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
            plt.show()




plot_button.on_click(plot_fun)

Discharge_pack_note = widgets.HTML(
    value= f" <b> Note: </b> the Standard Pack is calculated using the Autopopulate values above. The Autopopulate fields aren't similar to the other values this won't yield a meaningful result!")



display(V_cut_off_Widget, T_SOD_Switch, Current_Dist, Charge_SOD, Discharge_cells, Discharge_C_rate, Discharge_pack_note, Discharge_Pack, plot_button, output)
# Working

BoundedFloatText(value=1.8, description='Voltage Cutoff', layout=Layout(width='225px'), max=3.6, style=Descrip…

Dropdown(description='SOD or Time', layout=Layout(width='225px'), options=(('State of Discharge', 0), ('Time',…

Checkbox(value=False, description='Current Distribution in each String [All Currents]', indent=False)

Checkbox(value=False, description='String Charge vs Pack SOD for each String [All Currents]', indent=False)

Checkbox(value=False, description='Individual Cell Voltages [Only Shows lowest Current]', indent=False)

Checkbox(value=False, description='Impact of Current on Pack Voltage', indent=False)

HTML(value=" <b> Note: </b> the Standard Pack is calculated using the Autopopulate values above. The Autopopul…

Checkbox(value=False, description='Pack vs Standard Pack', indent=False)

Button(description='Plot the Selection', icon='check', style=ButtonStyle(), tooltip='Click me')

Output()

### Citations:
Battery Paramaters were obtained from the Energizer Website:
- $NMH$: https://data.energizer.com/pdfs/nh15-2000gl1220.pdf
- $LiFeS_2$: https://data.energizer.com/pdfs/L92GL0725.pdf
 
Discharge Curves for the following chemistries were obtained from Ziese et al [https://doi.org/10.1016/j.est.2020.101463] <br>
- $LFP$


Finalized May 3rd, 2026 \
Code written by Ronan Fiat in Collaboration with Professor Eric Stuve at the University of Washington 

In [8]:
#dir()

#### How it works
WIP!